# Notebook 05: Object Detection & Defect Localization with YOLO

**Objective:** Deep dive into the theory, mathematics, and implementation of real-time object detection for industrial quality inspection.

---

### Core Concepts Covered:
1. **Classification vs. Localization:** Why predicting $(x_{\text{center}}, y_{\text{center}}, w, h)$ and class probabilities requires specialized multi-task architectures.
2. **Anchor-Free Detection & TaskAlignedAssigner:** How modern YOLO (YOLOv8) eliminates manual anchor box clustering.
3. **Loss Formulations:** Complete IoU (CIoU) Loss + Distribution Focal Loss (DFL) vs. L1/L2 bounding box regression.
4. **Non-Maximum Suppression (NMS):** Mathematical mechanics and step-by-step visual demonstration.
5. **Evaluation Metrics:** Understanding mAP@50 (VOC standard) vs. mAP@50:95 (COCO standard).
6. **Inspection Pipeline:** Running the production `YOLODetector` inference engine on steel surface defects.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.detection_metrics import compute_iou_voc, non_maximum_suppression, calculate_average_precision
from src.inference.detector_inference import YOLODetector, DEFECT_COLOR_PALETTE, DetectionBox

print("Detection modules successfully imported!")

## 1. Why Anchor-Free YOLOv8?

Earlier YOLO architectures (YOLOv2 through YOLOv7) were **anchor-based**:
- Required pre-clustering dataset bounding boxes with k-means into 9 predefined anchor aspect ratios.
- Model predicted offsets $(\Delta x, \Delta y, \Delta w, \Delta h)$ relative to these preset anchors.
- Fragile to small or irregularly shaped industrial cracks (e.g., long narrow scratches or tiny inclusions).

**YOLOv8 is Anchor-Free:**
- The network directly predicts the distance from each feature map grid point $(x_c, y_c)$ to the 4 boundaries of the bounding box: $(l, t, r, b)$ (left, top, right, bottom).
- Uses **TaskAlignedAssigner (TAL)**: Dynamically matches ground truth to predictions by jointly ranking classification confidence and bounding box IoU alignment:
  $$\text{Alignment Metric} = t = s^\alpha \times \text{IoU}^\beta$$
  where $s$ is classification score, and $\alpha, \beta$ control the trade-off.
- Eliminates hyperparameter tuning for anchor sizes.

## 2. Loss Functions: Why Not L1 or Smooth L1 Loss?

Suppose a ground-truth box is $20 \times 20$ pixels (a small inclusion), and the prediction is $30 \times 30$ pixels.
The absolute error is $|30 - 20| = 10$ pixels ($50\%$ relative error).

Now suppose another ground-truth box is $500 \times 500$ pixels (a large patch), and the prediction is $510 \times 510$ pixels.
The absolute error is also $10$ pixels (only $2\%$ relative error).

**L1 and Smooth L1 treat both 10-pixel errors identically!**
This severely harms small defect detection.

**Complete IoU (CIoU) Loss:**
$$\mathcal{L}_{\text{CIoU}} = 1 - \text{IoU} + \frac{\rho^2(b, b^{\text{gt}})}{c^2} + \alpha v$$
- $\text{IoU}$: Scale-invariant overlap metric.
- $\frac{\rho^2(b, b^{\text{gt}})}{c^2}$: Normalized Euclidean distance between box center points relative to diagonal length $c$ of the smallest enclosing box.
- $\alpha v$: Aspect ratio consistency penalty.

In [ ]:
# Interactive Demonstration: Scale Invariance of IoU vs L1 Error
small_gt = (0, 0, 20, 20)
small_pred = (0, 0, 30, 30)

large_gt = (0, 0, 500, 500)
large_pred = (0, 0, 510, 510)

iou_small = compute_iou_voc(small_gt, small_pred)
iou_large = compute_iou_voc(large_gt, large_pred)

print(f"Small defect (20x20 vs 30x30): L1 Error = 10px | IoU = {iou_small:.4f}")
print(f"Large defect (500x500 vs 510x510): L1 Error = 10px | IoU = {iou_large:.4f}")
print("Notice how IoU accurately reflects the dramatic difference in defect alignment!")

## 3. Non-Maximum Suppression (NMS) from First Principles

During forward pass, convolutional heads predict multiple overlapping bounding boxes around the same defect.
NMS greedily selects the box with the highest confidence and eliminates redundant boxes whose overlap exceeds the IoU threshold.

In [ ]:
# Simulate dense candidate proposals around a single defect
candidate_boxes = [
    (30.0, 40.0, 110.0, 130.0),  # True center
    (32.0, 42.0, 108.0, 128.0),  # Near duplicate 1
    (28.0, 38.0, 112.0, 132.0),  # Near duplicate 2
    (140.0, 150.0, 185.0, 195.0) # Separate second defect
]
candidate_scores = [0.94, 0.82, 0.76, 0.89]
candidate_classes = [0, 0, 0, 0] # All predicted as crazing

surviving_indices = non_maximum_suppression(
    boxes=candidate_boxes,
    scores=candidate_scores,
    classes=candidate_classes,
    iou_threshold=0.45,
    conf_threshold=0.25
)

print(f"Original proposals: {len(candidate_boxes)}")
print(f"Surviving indices after NMS: {surviving_indices}")
print(f"Remaining boxes: {[candidate_boxes[i] for i in surviving_indices]}")

## 4. Visualizing Verified NEU YOLO Annotations

Let's load a training image and display its normalized YOLO bounding box conversions.

In [ ]:
yolo_img_dir = PROJECT_ROOT / "datasets" / "annotations" / "yolo" / "images" / "train"
yolo_lbl_dir = PROJECT_ROOT / "datasets" / "annotations" / "yolo" / "labels" / "train"

sample_img_paths = list(yolo_img_dir.glob("*.jpg"))[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, img_path in enumerate(sample_img_paths):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    lbl_path = yolo_lbl_dir / f"{img_path.stem}.txt"
    if lbl_path.exists():
        with open(lbl_path, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = line.strip().split()
            cls_id, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            # Convert normalized back to absolute pixels for visualization
            xmin = int((xc - bw / 2.0) * w)
            ymin = int((yc - bh / 2.0) * h)
            xmax = int((xc + bw / 2.0) * w)
            ymax = int((yc + bh / 2.0) * h)
            cv2.rectangle(img_rgb, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)
    
    axes[idx].imshow(img_rgb)
    axes[idx].set_title(f"{img_path.name}", fontsize=11)
    axes[idx].axis("off")

plt.suptitle("Ground-Truth Bounding Boxes on Converted YOLO Dataset", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. End-to-End Detector Inference Engine

Here we test our production `YOLODetector` inference wrapper with pre-trained weights.

In [ ]:
# Initialize detector with YOLOv8n
detector = YOLODetector(model_path="yolov8n.pt", device="cpu")

# Test on a sample image
test_img_path = sample_img_paths[0]
response = detector.predict(str(test_img_path))

print(f"Image Size         : {response.image_width}x{response.image_height}")
print(f"Inference Latency  : {response.inference_time_ms:.2f} ms")
print(f"Number of Defects  : {response.num_detections}")
print(f"Defective Status   : {response.is_defective}")